<a href="https://colab.research.google.com/github/GlobalFishingWatch/gfw-api-python-client/blob/develop/notebooks/usage-guides/events-api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Events API

This guide provides detailed instructions on how to use the [gfw-api-python-client](https://github.com/GlobalFishingWatch/gfw-api-python-client) to access information about various activities of a vessel, including fishing activity, encounters, port visits, loitering, and gaps in AIS reporting through the Global Fishing Watch [Events API](https://globalfishingwatch.org/our-apis/documentation#events-api).

**Note:** See the [Datasets](https://globalfishingwatch.org/our-apis/documentation#api-dataset), [Events Data Caveats](https://globalfishingwatch.org/our-apis/documentation#how-are-the-events-estimated), and [Terms of Use](https://globalfishingwatch.org/our-apis/documentation#terms-of-use) pages in the [GFW API documentation](https://globalfishingwatch.org/our-apis/documentation#introduction) for details on GFW data, API licenses, and rate limits.

## Prerequisites

Before using the `gfw-api-python-client`, ensure it is installed (see the [Getting Started](https://globalfishingwatch.github.io/gfw-api-python-client/getting-started.html) guide) and that you have obtained an API access token from the [Global Fishing Watch API portal](https://globalfishingwatch.org/our-apis/tokens).

## Installation

The `gfw-api-python-client` can be easily installed using pip:

In [1]:
# %pip install gfw-api-python-client

## Usage

Import and use `gfw-api-python-client` in your Python codes

In [2]:
import os

import geopandas as gpd

import gfwapiclient as gfw

In [3]:
try:
    from google.colab import userdata

    access_token = userdata.get("GFW_API_ACCESS_TOKEN")
except Exception:
    access_token = os.environ.get("GFW_API_ACCESS_TOKEN")

access_token = access_token or "<PASTE_YOUR_GFW_API_ACCESS_TOKEN_HERE>"

In [4]:
gfw_client = gfw.Client(
    access_token=access_token,
)

## Retrieving All Events from Predefined Region (`get_all_events`)

**Note:** See how to use the [Reference Data API - Usage Guides](https://globalfishingwatch.github.io/gfw-api-python-client/usage-guides/references-data-api.html) to obtain and filter predefined [**Regions of Interest (ROIs)**](https://globalfishingwatch.org/our-apis/documentation#regions), such as Exclusive Economic Zones (**EEZs**), Marine Protected Areas (**MPAs**), and Regional Fisheries Management Organizations (**RFMOs**).

In [5]:
eez_rois_result = await gfw_client.references.get_eez_regions(iso3="CHN")
chn_eez_roi = eez_rois_result.data()[0]

In [6]:
chn_eez_roi.id, chn_eez_roi.dataset, chn_eez_roi.label, chn_eez_roi.iso3

('8486', 'public-eez-areas', 'Chinese Exclusive Economic Zone', 'CHN')

In [7]:
events_result = await gfw_client.events.get_all_events(
    datasets=["public-global-fishing-events:latest"],
    start_date="2017-01-01",
    end_date="2017-01-31",
    region=chn_eez_roi,
    limit=5,
)

### Access the list of event as Pydantic models

In [8]:
events_data = events_result.data()

In [9]:
event = events_data[-1]

In [10]:
event.id, event.type, event.vessel.id

('54e1b8739c8ef032f2384e866b56077b',
 'fishing',
 'de2fb30db-b118-8a4e-edac-3764639a0d9e')

### Access the events as a DataFrame

In [11]:
events_df = events_result.df()

In [12]:
events_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   start         5 non-null      datetime64[us, UTC]
 1   end           5 non-null      datetime64[us, UTC]
 2   id            5 non-null      str                
 3   type          5 non-null      str                
 4   position      5 non-null      object             
 5   regions       5 non-null      object             
 6   bounding_box  5 non-null      object             
 7   distances     5 non-null      object             
 8   vessel        5 non-null      object             
 9   encounter     0 non-null      object             
 10  fishing       5 non-null      object             
 11  gap           0 non-null      object             
 12  loitering     0 non-null      object             
 13  port_visit    0 non-null      object             
dtypes: datetime64[us, UTC](2)

In [13]:
events_df.head()

,start,end,id,type,position,regions,bounding_box,distances,vessel,encounter,fishing,gap,loitering,port_visit
0,2016-12-28 22:53:16+00:00,2017-01-01 05:17:36+00:00,05cb42b98c7fcbb807527ef6749bdeda,fishing,"{'lat': 38.5911, 'lon': 118.3242}","{'mpa': [], 'eez': ['8486'], 'rfmo': ['PICES',...","[118.1391833333, 38.4701833333, 118.5858933333...","{'start_distance_from_shore_km': 31.0, 'end_di...",{'id': '1460ebe3f-fe57-05ee-5df7-803252df3983'...,None,"{'total_distance_km': 352.0775271758605, 'aver...",None,None,None
1,2016-12-29 05:52:08+00:00,2017-01-09 23:03:41+00:00,d0f7824acc0120713b806510205335c0,fishing,"{'lat': 38.7679, 'lon': 122.9222}","{'mpa': [], 'eez': ['8486'], 'rfmo': ['ACAP', ...","[122.8795516667, 38.73512, 122.967835, 38.8140...","{'start_distance_from_shore_km': 32.0, 'end_di...",{'id': 'b779f3880-0948-e042-79ac-7075cae0834a'...,None,"{'total_distance_km': 890.0985273915185, 'aver...",None,None,None
2,2016-12-29 06:02:29+00:00,2017-01-07 15:27:55+00:00,2e3edcbcf68a0be70c020d564a5bee3e,fishing,"{'lat': 38.8054, 'lon': 122.9166}","{'mpa': [], 'eez': ['8486'], 'rfmo': ['WCPFC',...","[122.8849333333, 38.7806466667, 122.9357066667...","{'start_distance_from_shore_km': 28.0, 'end_di...",{'id': 'b36e8c96e-e858-c54e-fdd5-ddd9f12e446e'...,None,"{'total_distance_km': 700.8972967886531, 'aver...",None,None,None
3,2016-12-29 06:20:32+00:00,2017-01-08 00:45:54+00:00,3d57f4311f1caebe1479057da1fb2d66,fishing,"{'lat': 38.7666, 'lon': 122.9176}","{'mpa': [], 'eez': ['8486'], 'rfmo': ['WCPFC',...","[122.8741416667, 38.7374516667, 122.9684316667...","{'start_distance_from_shore_km': 32.0, 'end_di...",{'id': 'c5da36777-79a9-4eb8-09c6-7db3c7e8bd4f'...,None,"{'total_distance_km': 730.167140184076, 'avera...",None,None,None
4,2016-12-29 12:50:28+00:00,2017-01-01 12:35:12+00:00,54e1b8739c8ef032f2384e866b56077b,fishing,"{'lat': 38.9222, 'lon': 120.8962}","{'mpa': [], 'eez': ['8486'], 'rfmo': ['IWC', '...","[120.8618383333, 38.86113, 120.9324016667, 38....","{'start_distance_from_shore_km': 14.0, 'end_di...",{'id': 'de2fb30db-b118-8a4e-edac-3764639a0d9e'...,None,"{'total_distance_km': 251.8224420985952, 'aver...",None,None,None


## Retrieving All Events from Custom Region (`get_all_events`)

**Note:** Custom region can either a path to a spatial file (e.g., GeoJSON, Shapefile, etc.), GeoJSON-like object (e.g., JSON string, dictionary, `geopandas.GeoDataFrame`, `shapely`, an object implementing `__geo_interface__` etc.) or `GeoJson` model instance. Spatial files are loaded using [geopandas.read_file](https://geopandas.org/en/stable/docs/reference/api/geopandas.read_file.html) and supported formats depend on a properly configured [geopandas/GDAL installation](https://geopandas.org/en/stable/getting_started/install.html#installing-with-pip).

In [14]:
filename = "https://raw.githubusercontent.com/GlobalFishingWatch/gfw-api-python-client/refs/heads/develop/tests/fixtures/events/geometry/geometry.shp"

In [15]:
custom_roi_gdf = gpd.read_file(filename)

In [16]:
custom_events_result = await gfw_client.events.get_all_events(
    datasets=["public-global-fishing-events:latest"],
    start_date="2017-01-01",
    end_date="2017-01-31",
    geometry=custom_roi_gdf,
    limit=5,
)

### Access the list of event as Pydantic models

In [17]:
custom_events_data = custom_events_result.data()

In [18]:
custom_event = custom_events_data[-1]

In [19]:
custom_event.id, custom_event.type, custom_event.vessel.id

('5c03609c64d96c6ca5bfaaca0e9d9b6c',
 'fishing',
 'c01e0a0d2-20d9-7cc6-e04e-449dae2fbd95')

### Access the events as a DataFrame

In [20]:
custom_events_df = custom_events_result.df()

In [21]:
custom_events_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   start         5 non-null      datetime64[us, UTC]
 1   end           5 non-null      datetime64[us, UTC]
 2   id            5 non-null      str                
 3   type          5 non-null      str                
 4   position      5 non-null      object             
 5   regions       5 non-null      object             
 6   bounding_box  5 non-null      object             
 7   distances     5 non-null      object             
 8   vessel        5 non-null      object             
 9   encounter     0 non-null      object             
 10  fishing       5 non-null      object             
 11  gap           0 non-null      object             
 12  loitering     0 non-null      object             
 13  port_visit    0 non-null      object             
dtypes: datetime64[us, UTC](2)

In [22]:
custom_events_df.head()

,start,end,id,type,position,regions,bounding_box,distances,vessel,encounter,fishing,gap,loitering,port_visit
0,2016-12-30 05:09:54+00:00,2017-01-01 20:03:59+00:00,d6ee1cb7351fcb2c6b6e3b675335cb2c,fishing,"{'lat': 27.1871, 'lon': 121.3279}","{'mpa': [], 'eez': ['8486'], 'rfmo': ['WCPFC',...","[121.139415, 27.0787716667, 121.4224816667, 27...","{'start_distance_from_shore_km': 40.0, 'end_di...",{'id': '4de4289ac-cb8b-81de-4799-05b592602bf1'...,None,"{'total_distance_km': 208.03116341478372, 'ave...",None,None,None
1,2016-12-30 05:21:43+00:00,2017-01-02 03:17:00+00:00,b5b0895669f4123b5da0eed20210342d,fishing,"{'lat': 27.0703, 'lon': 121.2056}","{'mpa': [], 'eez': ['8486'], 'rfmo': ['APFIC',...","[121.0888466667, 26.9642966667, 121.34091, 27....","{'start_distance_from_shore_km': 38.0, 'end_di...",{'id': '3ee872640-08b1-04cf-75f8-d90932461f6f'...,None,"{'total_distance_km': 221.3299351192625, 'aver...",None,None,None
2,2016-12-30 06:26:05+00:00,2017-01-02 02:38:20+00:00,088ef328a8fe54bd46e6fccd539ae55a,fishing,"{'lat': 27.0929, 'lon': 121.2196}","{'mpa': [], 'eez': ['8486'], 'rfmo': ['APFIC',...","[121.1208266667, 27.00363, 121.3079333333, 27....","{'start_distance_from_shore_km': 40.0, 'end_di...",{'id': '55cf89775-54e1-89aa-6e92-fef0b8deb419'...,None,"{'total_distance_km': 224.6395440027435, 'aver...",None,None,None
3,2016-12-30 07:28:06+00:00,2017-01-02 03:49:40+00:00,8326eb5fb66215de739f0c889a9b6a8e,fishing,"{'lat': 27.0865, 'lon': 121.2232}","{'mpa': [], 'eez': ['8486'], 'rfmo': ['WCPFC',...","[121.1017333333, 26.9795383333, 121.3309066667...","{'start_distance_from_shore_km': 38.0, 'end_di...",{'id': '3cedf7bd9-9800-1fc2-11d7-7985205d926e'...,None,"{'total_distance_km': 235.3635712625424, 'aver...",None,None,None
4,2016-12-30 07:46:18+00:00,2017-01-02 04:28:40+00:00,5c03609c64d96c6ca5bfaaca0e9d9b6c,fishing,"{'lat': 27.1714, 'lon': 121.3037}","{'mpa': [], 'eez': ['8486'], 'rfmo': ['ACAP', ...","[121.1050983333, 27.0637933333, 121.43991, 27....","{'start_distance_from_shore_km': 39.0, 'end_di...",{'id': 'c01e0a0d2-20d9-7cc6-e04e-449dae2fbd95'...,None,"{'total_distance_km': 223.68434789759047, 'ave...",None,None,None


## Retrieving a Single Event by ID (`get_event_by_id`)

In [23]:
event_result = await gfw_client.events.get_event_by_id(
    id="c2f0967e061f99a01793edac065de003",
    dataset="public-global-port-visits-events:latest",
)

### Access the event as Pydantic model

In [24]:
event = event_result.data()

In [25]:
event.id, event.type, event.vessel.id

('c2f0967e061f99a01793edac065de003',
 'port_visit',
 '8c7304226-6c71-edbe-0b63-c246734b3c01')

### Access the event as a DataFrame

In [26]:
event_df = event_result.df()

In [27]:
event_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   start         1 non-null      datetime64[us, UTC]
 1   end           1 non-null      datetime64[us, UTC]
 2   id            1 non-null      str                
 3   type          1 non-null      str                
 4   position      1 non-null      object             
 5   regions       1 non-null      object             
 6   bounding_box  1 non-null      object             
 7   distances     1 non-null      object             
 8   vessel        1 non-null      object             
 9   encounter     0 non-null      object             
 10  fishing       0 non-null      object             
 11  gap           0 non-null      object             
 12  loitering     0 non-null      object             
 13  port_visit    1 non-null      object             
dtypes: datetime64[us, UTC](2)

In [28]:
event_df.head()

,start,end,id,type,position,regions,bounding_box,distances,vessel,encounter,fishing,gap,loitering,port_visit
0,2020-01-26 05:52:47+00:00,2020-01-29 14:39:33+00:00,c2f0967e061f99a01793edac065de003,port_visit,"{'lat': 20.7288, 'lon': -17.0148}","{'mpa': [], 'eez': ['8369'], 'rfmo': ['IWC', '...","[-17.014774393446658, 20.72879719687954, -17.0...","{'start_distance_from_shore_km': 7.0, 'end_dis...",{'id': '8c7304226-6c71-edbe-0b63-c246734b3c01'...,None,None,None,None,{'visit_id': '38affb3e7bdc67e9c0c2e7e8f3b08da2...


## Getting Event Statistics Worldwide (`get_events_stats`)

In [29]:
worldwide_event_stats_result = await gfw_client.events.get_events_stats(
    datasets=["public-global-encounters-events:latest"],
    encounter_types=["CARRIER-FISHING", "FISHING-CARRIER"],
    vessel_types=["CARRIER"],
    start_date="2018-01-01",
    end_date="2023-01-31",
    timeseries_interval="YEAR",
    flags=["RUS"],
    duration=60,
)

### Access the statistics as Pydantic models

In [30]:
worldwide_event_stat = worldwide_event_stats_result.data()

In [31]:
(
    worldwide_event_stat.num_events,
    worldwide_event_stat.num_flags,
    worldwide_event_stat.num_vessels,
)

(24819, 1, 194)

### Access the statistics as a DataFrame

In [32]:
worldwide_event_stat_df = worldwide_event_stats_result.df()

In [33]:
worldwide_event_stat_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   num_events   1 non-null      int64 
 1   num_flags    1 non-null      int64 
 2   num_vessels  1 non-null      int64 
 3   flags        1 non-null      object
 4   timeseries   1 non-null      object
dtypes: int64(3), object(2)
memory usage: 172.0+ bytes


In [34]:
worldwide_event_stat_df

,num_events,num_flags,num_vessels,flags,timeseries
0,24819,1,194,[RUS],"[{'date': 2018-01-01 00:00:00+00:00, 'value': ..."


## Getting Event Statistics from Predefined Region (`get_events_stats`)

**Note:** See how to use the [Reference Data API - Usage Guides](https://globalfishingwatch.github.io/gfw-api-python-client/usage-guides/references-data-api.html) to obtain and filter predefined [**Regions of Interest (ROIs)**](https://globalfishingwatch.org/our-apis/documentation#regions), such as Exclusive Economic Zones (**EEZs**), Marine Protected Areas (**MPAs**), and Regional Fisheries Management Organizations (**RFMOs**).

In [35]:
eez_rois_result = await gfw_client.references.get_eez_regions(iso3="SEN")
sen_eez_roi = eez_rois_result.data()[0]

In [36]:
sen_eez_roi.id, sen_eez_roi.dataset, sen_eez_roi.label, sen_eez_roi.iso3

('8371', 'public-eez-areas', 'Senegalese Exclusive Economic Zone', 'SEN')

In [37]:
predefined_event_stats_result = await gfw_client.events.get_events_stats(
    datasets=["public-global-port-visits-events:latest"],
    start_date="2018-01-01",
    end_date="2019-01-31",
    timeseries_interval="YEAR",
    region=sen_eez_roi,
    confidences=["3", "4"],
)

### Access the statistics as Pydantic models

In [38]:
predefined_event_stat = predefined_event_stats_result.data()

In [39]:
(
    predefined_event_stat.num_events,
    predefined_event_stat.num_flags,
    predefined_event_stat.num_vessels,
)

(4528, 75, 1464)

### Access the statistics as a DataFrame

In [40]:
predefined_event_stat_df = predefined_event_stats_result.df()

In [41]:
predefined_event_stat_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   num_events   1 non-null      int64 
 1   num_flags    1 non-null      int64 
 2   num_vessels  1 non-null      int64 
 3   flags        1 non-null      object
 4   timeseries   1 non-null      object
dtypes: int64(3), object(2)
memory usage: 172.0+ bytes


In [42]:
predefined_event_stat_df.head()

,num_events,num_flags,num_vessels,flags,timeseries
0,4528,75,1464,"[, PAN, BHS, CYP, SEN, SGP, CHN, ITA, RUS, ESP...","[{'date': 2018-01-01 00:00:00+00:00, 'value': ..."


## Getting Event Statistics from Custom Region (`get_events_stats`)

**Note:** Custom region can either a path to a spatial file (e.g., GeoJSON, Shapefile, etc.), GeoJSON-like object (e.g., JSON string, dictionary, `geopandas.GeoDataFrame`, `shapely`, an object implementing `__geo_interface__` etc.) or `GeoJson` model instance. Spatial files are loaded using [geopandas.read_file](https://geopandas.org/en/stable/docs/reference/api/geopandas.read_file.html) and supported formats depend on a properly configured [geopandas/GDAL installation](https://geopandas.org/en/stable/getting_started/install.html#installing-with-pip).

In [43]:
filename = "https://raw.githubusercontent.com/GlobalFishingWatch/gfw-api-python-client/refs/heads/develop/tests/fixtures/events/geometry/geometry.shp"

In [44]:
custom_stats_roi_gdf = gpd.read_file(filename)

In [45]:
custom_event_stats_result = await gfw_client.events.get_events_stats(
    datasets=["public-global-port-visits-events:latest"],
    start_date="2018-01-01",
    end_date="2019-01-31",
    timeseries_interval="YEAR",
    geometry=custom_stats_roi_gdf,
    confidences=["3", "4"],
)

### Access the statistics as Pydantic models

In [46]:
custom_event_stat = custom_event_stats_result.data()

In [47]:
(
    custom_event_stat.num_events,
    custom_event_stat.num_flags,
    custom_event_stat.num_vessels,
)

(301548, 162, 40996)

### Access the statistics as a DataFrame

In [48]:
custom_event_stat_df = custom_event_stats_result.df()

In [49]:
custom_event_stat_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   num_events   1 non-null      int64 
 1   num_flags    1 non-null      int64 
 2   num_vessels  1 non-null      int64 
 3   flags        1 non-null      object
 4   timeseries   1 non-null      object
dtypes: int64(3), object(2)
memory usage: 172.0+ bytes


In [50]:
custom_event_stat_df.head()

,num_events,num_flags,num_vessels,flags,timeseries
0,301548,162,40996,"[CHN, , BTN, CCK, VAT, GRL, FRO, BLR, GEO, ALB...","[{'date': 2018-01-01 00:00:00+00:00, 'value': ..."
